# Project 1 V3: Transformer with a Pretrained BERT Backbone

V3 keeps the data, the split, the two training phases and the evaluation of V2. It replaces the MLP with a **transformer** whose layers are initialised from a **pretrained BERT encoder**.

- **Phase 1** trains on the extended dataset (provided + LP-generated rows).
- **Fine-tuning** then trains on the provided rows only.

The design follows three findings:
1. The m-height is a maximum over a finite set of codewords, each built from a combination of columns of $G$ (Roth et al., 2026). Self-attention compares columns with each other, which suits this structure. Attention-based networks are also the standard way to learn LP properties (Chen et al., ICLR 2023).
2. Transformers pretrained on text transfer to numerical tasks (Lu et al., *Frozen Pretrained Transformers as Universal Computation Engines*, AAAI 2022).
3. The m-height does not change when the columns of $G$ are permuted or have their signs flipped, or when the rows of $G$ are permuted or have their signs flipped. The model therefore reads the columns as an unordered set: there are no position embeddings. During training, rows and signs are randomly permuted and flipped.

The pretrained weights come from Google's compact BERT models (Turc et al., 2019, Apache-2.0), loaded from the original TensorFlow checkpoints into standard Keras layers. The saved model contains only built-in Keras layers.


In [ ]:
# Install dependencies (Google Colab only; local/NERSC runs use the project environment).

import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" "tensorflow==2.21.0" "keras==3.15.1"


## Load the Data

The extended files contain the provided rows first, followed by the generated rows. `N_PROVIDED`, the number of provided rows, marks the boundary. The provided rows are the only ones used for fine-tuning and for model selection.


In [ ]:
import os
import pickle
import numpy as np

NUMPY_MAJOR = int(np.__version__.split(".")[0])


class NumpyCompatUnpickler(pickle.Unpickler):
    # NumPy 2 pickles reference numpy._core; NumPy 1.x only provides numpy.core.
    def find_class(self, module, name):
        if NUMPY_MAJOR < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as file:
        return NumpyCompatUnpickler(file).load()


# Pre-training uses the extended data; its first N_PROVIDED rows are the provided data.
DATASET = "extended"
features = load_pickle("data/CSCE-636-Project-1-Train-Extended-n_k_m_P")
labels = load_pickle("data/CSCE-636-Project-1-Train-Extended-mHeights")
print(f"Dataset: {DATASET}")

N_PROVIDED = len(load_pickle("data/CSCE-636-Project-1-Train-mHeights"))

if len(features) != len(labels):
    raise ValueError(
        f"Sample/label count mismatch: {len(features)} samples, {len(labels)} labels."
    )

first_five_features = features[:5]
first_five_labels = labels[:5]
print(f"Loaded {len(features):,} samples and {len(labels):,} corresponding labels.")

# Print the first five samples and their corresponding labels.
for index, ((n, k, m, P), m_height) in enumerate(
    zip(first_five_features, first_five_labels)
):
    if n != 9 or k not in (4, 5, 6) or not (2 <= m <= n - k):
        raise ValueError(f"Sample at index {index} has invalid n, k, or m.")
    if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
        raise ValueError(f"Sample at index {index}: expected a NumPy array of shape {(k, n - k)}.")

    print("=" * 64)
    print(f"Sample {index + 1} (list index {index})")
    print(f"n = {n}, k = {k}, m = {m}")
    print(f"P.shape = {P.shape}; expected (k, n-k) = {(k, n - k)}")
    print(f"P dtype: {P.dtype}")
    print("P contents:")
    with np.printoptions(threshold=np.inf, floatmode="unique", linewidth=120):
        print(P)
    print(f"True m-height = {m_height!r}")



## Configuration

The network predicts $z=\log_2(\hat y)$, bounded to `[0, 100]`, and $\hat y=2^z$.

| Setting | Default | Environment variable |
|---|---|---|
| Backbone | `bert-small` (4 layers, width 512) or `bert-mini` (4 layers, width 256) | `MHEIGHT_BACKBONE` |
| Initialisation | `pretrained` (BERT weights) or `scratch` (random, as a control) | `MHEIGHT_INIT` |
| Freeze attention/feed-forward weights | `0` (train everything) or `1` (train only embeddings, LayerNorms and head, as in Lu et al.) | `MHEIGHT_FREEZE` |
| Random row permutations and sign flips in training | `1` | `MHEIGHT_AUGMENT` |
| Dropout | 0.1 (as in BERT) | `MHEIGHT_DROPOUT` |
| Phase 1 epochs / learning rate | 40 / 2e-4 | `MHEIGHT_PHASE1_EPOCHS` / `MHEIGHT_LR` |
| Fine-tuning epochs / learning rate | 80 / 5e-5 | `MHEIGHT_FINETUNE_EPOCHS` / `MHEIGHT_FINETUNE_LR` |
| Output directory | `artifacts/dnn_v3_<backbone>_<init>` | `MHEIGHT_RUN` |

In [ ]:
import hashlib
import os
import json
import pickle
import numpy as np
from pathlib import Path
import tensorflow as tf

SEED = 636
BATCH_SIZE = 4096
INFERENCE_BATCH_SIZE = 1024
PHASE1_EPOCHS = int(os.environ.get("MHEIGHT_PHASE1_EPOCHS", 40))
FINETUNE_EPOCHS = int(os.environ.get("MHEIGHT_FINETUNE_EPOCHS", 80))
LEARNING_RATE = float(os.environ.get("MHEIGHT_LR", 2e-4))
FINETUNE_LEARNING_RATE = float(os.environ.get("MHEIGHT_FINETUNE_LR", 5e-5))
BACKBONE = os.environ.get("MHEIGHT_BACKBONE", "bert-small")
INIT = os.environ.get("MHEIGHT_INIT", "pretrained")
FREEZE_BACKBONE = os.environ.get("MHEIGHT_FREEZE", "0") == "1"
AUGMENT = os.environ.get("MHEIGHT_AUGMENT", "1") == "1"
DROPOUT = float(os.environ.get("MHEIGHT_DROPOUT", 0.1))
if INIT not in ("pretrained", "scratch"):
    raise ValueError("MHEIGHT_INIT must be 'pretrained' or 'scratch'")
MAX_LOG2_HEIGHT = 100.0
ARTIFACT_DIR = Path("artifacts") / os.environ.get("MHEIGHT_RUN", f"dnn_v3_{BACKBONE}_{INIT}")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# Model after phase 1 (extended data); "pretrained" is reserved for the BERT weights.
PHASE1_PATH = ARTIFACT_DIR / "phase1_model.keras"
# The fine-tuned model is the final model.
MODEL_PATH = ARTIFACT_DIR / "best_model.keras"
PARAMETER_SETTINGS = [(9, k, m) for k in (4, 5, 6) for m in range(2, 9 - k + 1)]

# Set TensorFlow to be deterministic and reproducible.
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic operations in TensorFlow to ensure reproducibility.
tf.config.experimental.enable_op_determinism()

# Enable GPU memory growth to avoid allocating all GPU memory at once.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

# Print TensorFlow version and available GPUs.
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print(f"Backbone: {BACKBONE} | init: {INIT} | frozen: {FREEZE_BACKBONE} | augment: {AUGMENT} | run: {ARTIFACT_DIR}")


## Input Formatting and Data Splitting

The model reads the full generator matrix $G = [I_k \mid P]$, placed in the upper-left corner of a zero-padded `6 × 9` array, together with `[n, k, m]`. Building $G$ only places the identity next to $P$. Zero rows do not change the code: they add nothing to any codeword. Fixed scaling occurs inside standard Keras `Rescaling` layers.

The same matrix can appear under several `m` values. Hence, a row-wise shuffle could place that matrix in both training and testing. Therefore, a seeded hash of `(n, k, P)` assigns each distinct matrix to training, validation, or testing with probabilities **60%, 20%, 20%**. Identical matrix contents always get the same assignment, even if stored as different NumPy objects. Actual row percentages are approximate because whole matrix groups stay together. The hash is used only for partitioning and is never fed into the model.

Nonfinite or below-one training labels have no valid finite log-loss target. We report and exclude such rows from supervised training and local evaluation.

In [ ]:
def pack_features(samples):
    # G = [I_k | P] in the upper-left corner of a zero-padded 6 x 9 array.
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.empty((len(samples), 3), dtype=np.float32)
    for index, (n, k, m, P) in enumerate(samples):
        if n != 9 or k not in (4, 5, 6) or not 2 <= m <= n - k:
            raise ValueError(f"Invalid parameters at row {index}: {(n, k, m)}")
        if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
            raise ValueError(f"Invalid P shape at row {index}")
        generators[index, :k, :k] = np.eye(k, dtype=np.float32)
        generators[index, :k, k:n] = P
        parameters[index] = (n, k, m)
    if not np.isfinite(generators).all():
        raise ValueError("Input matrices must remain finite after float32 conversion")
    return {"generator": generators, "parameters": parameters}


def matrix_partitions(samples, seed):
    partitions = np.empty(len(samples), dtype=np.uint8)
    object_cache = {}
    key = int(seed).to_bytes(8, "little")
    for index, (n, k, m, P) in enumerate(samples):
        cache_key = (int(n), int(k), id(P))
        if cache_key not in object_cache:
            payload = bytes((int(n), int(k))) + np.asarray(P, dtype="<f8").tobytes(order="C")
            digest = hashlib.blake2b(payload, digest_size=8, key=key).digest()
            bucket = int.from_bytes(digest, "little") % 10
            object_cache[cache_key] = 0 if bucket < 6 else (1 if bucket < 8 else 2)
        partitions[index] = object_cache[cache_key]
    return partitions


In [ ]:
if len(features) != len(labels):
    raise ValueError("Feature and label counts differ")
raw_targets = np.asarray(labels, dtype=np.float64)
if raw_targets.shape != (len(features),):
    raise ValueError("Expected one scalar label per sample")
valid_target = np.isfinite(raw_targets) & (raw_targets >= 1.0)
print(f"Excluded {np.count_nonzero(~valid_target):,} invalid training labels")
if not np.any(valid_target):
    raise ValueError("No valid training labels")

targets_log2 = np.zeros((len(features), 1), dtype=np.float32)
targets_log2[valid_target, 0] = np.log2(raw_targets[valid_target])
if targets_log2[valid_target].max() > MAX_LOG2_HEIGHT:
    raise ValueError("A training target exceeds the model output range")
inputs = pack_features(features)
partition = matrix_partitions(features, SEED)
train_indices = np.flatnonzero(valid_target & (partition == 0))
validation_indices = np.flatnonzero(valid_target & (partition == 1))
test_indices = np.flatnonzero(valid_target & (partition == 2))

setting_ids = np.empty(len(features), dtype=np.int8)
for group_id, (n, k, m) in enumerate(PARAMETER_SETTINGS):
    mask = (inputs["parameters"][:, 1] == k) & (inputs["parameters"][:, 2] == m)
    setting_ids[mask] = group_id

splits = {"train": train_indices, "validation": validation_indices, "test": test_indices}
for name, indices in splits.items():
    if len(indices) == 0:
        raise ValueError(f"The {name} split is empty")
    print(f"{name:10s}: {len(indices):,} rows ({100 * len(indices) / valid_target.sum():.2f}%)")
print("\n(n, k, m)       train       validation       test")
for group_id, setting in enumerate(PARAMETER_SETTINGS):
    counts = [int(np.count_nonzero(setting_ids[indices] == group_id)) for indices in splits.values()]
    if min(counts) == 0:
        raise ValueError(f"Setting {setting} is missing from a split")
    print(f"{str(setting):12s} {counts[0]:10,d} {counts[1]:14,d} {counts[2]:10,d}")
np.savez_compressed(ARTIFACT_DIR / "split_indices.npz", **splits)

# Provided-data rows of each split: used for fine-tuning, model selection and reporting.
provided_train_indices = train_indices[train_indices < N_PROVIDED]
provided_validation_indices = validation_indices[validation_indices < N_PROVIDED]
provided_test_indices = test_indices[test_indices < N_PROVIDED]
print(f"\nProvided-data rows: train {len(provided_train_indices):,}, "
      f"validation {len(provided_validation_indices):,}, test {len(provided_test_indices):,}")


## DNN: Transformer over the Columns of G

**Tokens.** Each of the 9 columns of $G$ (a vector of 6 values after padding) is one token, embedded by a `Dense` layer. A tenth token is a `Dense` embedding of `[n, k, m]`, so attention knows which m-height is asked for. There are **no position embeddings**, so reordering the columns cannot change the output.

**Encoder.** Then come the BERT layers (post-LayerNorm, as in BERT). Each layer:
- multi-head self-attention, then dropout, then `LayerNorm(x + ·)`;
- a `Dense(intermediate, GELU)` → `Dense(width)` feed-forward block, then dropout, then `LayerNorm(x + ·)`.

**Pretrained weights.** With `MHEIGHT_INIT=pretrained`, the following are copied from the BERT checkpoint:
- the attention projections (query, key, value, output);
- the feed-forward layers;
- all LayerNorms, including the embedding LayerNorm.

The column and parameter embeddings and the output head are new, since BERT's word embeddings have no meaning here. With `MHEIGHT_FREEZE=1`, the copied attention and feed-forward weights stay fixed and only the rest is trained (the setting of Lu et al.).

**Output head.** It combines three summaries:
- the parameter token's output (kept by a `Cropping1D` layer);
- the maximum over tokens, which mirrors the "maximum over codewords" in the definition;
- the mean over tokens.

Training and validation rows are weighted inversely by their setting's frequency, so all 9 parameter settings count equally.


In [ ]:
import urllib.request

BACKBONES = {
    "bert-small": "google/bert_uncased_L-4_H-512_A-8",
    "bert-mini": "google/bert_uncased_L-4_H-256_A-4",
}


def backbone_files(name):
    """Download the original TensorFlow BERT checkpoint (Apache-2.0) once."""
    repository = BACKBONES[name]
    folder = Path("artifacts/pretrained") / repository.split("/")[1]
    folder.mkdir(parents=True, exist_ok=True)
    for file_name in ("config.json", "bert_model.ckpt.index", "bert_model.ckpt.data-00000-of-00001"):
        target = folder / file_name
        if not target.exists():
            url = f"https://huggingface.co/{repository}/resolve/main/{file_name}"
            print("Downloading", url)
            urllib.request.urlretrieve(url, target.with_name(file_name + ".tmp"))
            target.with_name(file_name + ".tmp").replace(target)
    return json.loads((folder / "config.json").read_text()), str(folder / "bert_model.ckpt")


def encoder_layer(x, index, config, dropout):
    width, heads = config["hidden_size"], config["num_attention_heads"]
    epsilon = config["layer_norm_eps"]
    attention = tf.keras.layers.MultiHeadAttention(
        num_heads=heads, key_dim=width // heads, dropout=dropout, name=f"layer{index}_attention")(x, x)
    attention = tf.keras.layers.Dropout(dropout)(attention)
    x = tf.keras.layers.LayerNormalization(epsilon=epsilon, name=f"layer{index}_attention_norm")(
        tf.keras.layers.Add()([x, attention]))
    hidden = tf.keras.layers.Dense(config["intermediate_size"], activation="gelu",
                                   name=f"layer{index}_intermediate")(x)
    hidden = tf.keras.layers.Dense(width, name=f"layer{index}_output")(hidden)
    hidden = tf.keras.layers.Dropout(dropout)(hidden)
    return tf.keras.layers.LayerNormalization(epsilon=epsilon, name=f"layer{index}_output_norm")(
        tf.keras.layers.Add()([x, hidden]))


def build_network(config, dropout=DROPOUT):
    width, epsilon = config["hidden_size"], config["layer_norm_eps"]
    generator_input = tf.keras.Input(shape=(6, 9), name="generator")
    parameter_input = tf.keras.Input(shape=(3,), name="parameters")
    columns = tf.keras.layers.Rescaling(1.0 / 100.0)(generator_input)
    columns = tf.keras.layers.Permute((2, 1))(columns)                  # 9 column tokens of 6 values
    columns = tf.keras.layers.Dense(width, name="column_embedding")(columns)
    parameters = tf.keras.layers.Rescaling(1.0 / 10.0)(parameter_input)
    parameters = tf.keras.layers.Dense(width, name="parameter_embedding")(parameters)
    parameters = tf.keras.layers.Reshape((1, width))(parameters)
    x = tf.keras.layers.Concatenate(axis=1)([parameters, columns])     # 10 tokens, no positions
    x = tf.keras.layers.LayerNormalization(epsilon=epsilon, name="embedding_norm")(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    for index in range(config["num_hidden_layers"]):
        x = encoder_layer(x, index, config, dropout)
    parameter_token = tf.keras.layers.Flatten()(tf.keras.layers.Cropping1D((0, 9))(x))
    summary = tf.keras.layers.Concatenate()([
        parameter_token,
        tf.keras.layers.GlobalMaxPooling1D()(x),
        tf.keras.layers.GlobalAveragePooling1D()(x),
    ])
    summary = tf.keras.layers.Dense(width, activation="gelu", name="head_hidden")(summary)
    log2_value = tf.keras.layers.Dense(
        1, activation="softplus", bias_initializer=tf.keras.initializers.Constant(8.0),
        name="head_output")(summary)
    output = tf.keras.layers.ReLU(max_value=MAX_LOG2_HEIGHT, name="log2_height")(log2_value)
    return tf.keras.Model(inputs={"generator": generator_input, "parameters": parameter_input},
                          outputs=output, name="mheight_transformer")


def load_bert_weights(network, config, checkpoint):
    """Copy BERT encoder weights into the matching Keras layers."""
    reader = tf.train.load_checkpoint(checkpoint)
    width, heads = config["hidden_size"], config["num_attention_heads"]
    head_size = width // heads

    def weight(name):
        return reader.get_tensor(name)

    def set_norm(layer_name, prefix):
        network.get_layer(layer_name).set_weights([weight(prefix + "/gamma"), weight(prefix + "/beta")])

    set_norm("embedding_norm", "bert/embeddings/LayerNorm")
    for index in range(config["num_hidden_layers"]):
        prefix = f"bert/encoder/layer_{index}"
        attention = network.get_layer(f"layer{index}_attention")
        # BERT splits each projection's output into heads as [head, head_size].
        for part, dense in (("query", attention.query_dense), ("key", attention.key_dense),
                            ("value", attention.value_dense)):
            dense.set_weights([
                weight(f"{prefix}/attention/self/{part}/kernel").reshape(width, heads, head_size),
                weight(f"{prefix}/attention/self/{part}/bias").reshape(heads, head_size),
            ])
        attention.output_dense.set_weights([
            weight(f"{prefix}/attention/output/dense/kernel").reshape(heads, head_size, width),
            weight(f"{prefix}/attention/output/dense/bias"),
        ])
        set_norm(f"layer{index}_attention_norm", f"{prefix}/attention/output/LayerNorm")
        network.get_layer(f"layer{index}_intermediate").set_weights([
            weight(f"{prefix}/intermediate/dense/kernel"), weight(f"{prefix}/intermediate/dense/bias")])
        network.get_layer(f"layer{index}_output").set_weights([
            weight(f"{prefix}/output/dense/kernel"), weight(f"{prefix}/output/dense/bias")])
        set_norm(f"layer{index}_output_norm", f"{prefix}/output/LayerNorm")


def compile_network(network, learning_rate):
    if FREEZE_BACKBONE:
        for layer in network.layers:
            if layer.name.endswith(("_attention", "_intermediate", "_output")) and layer.name.startswith("layer"):
                layer.trainable = False
    network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0),
        loss="mean_squared_error",
        # Sample weights already balance the loss; no additional metrics requested.
        weighted_metrics=[],
    )


backbone_config, backbone_checkpoint = backbone_files(BACKBONE)
model = build_network(backbone_config)
if INIT == "pretrained":
    load_bert_weights(model, backbone_config, backbone_checkpoint)
    print(f"Loaded pretrained {BACKBONES[BACKBONE]} encoder weights")
compile_network(model, LEARNING_RATE)
model.summary()


In [ ]:
def setting_weights(indices):
    counts = np.bincount(setting_ids[indices], minlength=len(PARAMETER_SETTINGS))
    if np.any(counts == 0):
        raise ValueError("Each parameter setting must be present in the weighted split")
    return (len(indices) / (len(PARAMETER_SETTINGS) * counts[setting_ids[indices]])).astype(np.float32)


def augment_generators(features, target, weight):
    """Random row permutations and row/column sign flips of G.

    These leave the code, and therefore every m-height, unchanged: permuting or negating
    rows of G keeps its row space, and negating a column negates one coordinate of every
    codeword. Column order needs no augmentation because the model ignores it.
    """
    generators = features["generator"]
    batch = tf.shape(generators)[0]
    row_order = tf.argsort(tf.random.uniform((batch, 6)), axis=1)
    generators = tf.gather(generators, row_order, batch_dims=1)
    row_signs = tf.where(tf.random.uniform((batch, 6, 1)) < 0.5, -1.0, 1.0)
    column_signs = tf.where(tf.random.uniform((batch, 1, 9)) < 0.5, -1.0, 1.0)
    features = dict(features, generator=generators * row_signs * column_signs)
    return features, target, weight


def make_dataset(indices, training=False):
    # Keep the dataset on host memory; only batches move to the training device.
    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((
            {name: values[indices] for name, values in inputs.items()},
            targets_log2[indices], setting_weights(indices),
        ))
    if training:
        dataset = dataset.shuffle(min(len(indices), 100_000), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    dataset = dataset.with_options(options).batch(BATCH_SIZE)
    if training and AUGMENT:
        dataset = dataset.map(augment_generators)
    return dataset.prefetch(1)


## Phase 1: Train on the Extended Data

Phase 1 uses all extended training rows. Its validation set is all extended validation rows. The best weights are restored at the end and saved as `phase1_model.keras`.


In [ ]:
def training_callbacks(checkpoint_path):
    return [
        tf.keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def save_history(history, name):
    if not np.isfinite(history.history["val_loss"]).any():
        raise RuntimeError("Training produced no finite validation loss; do not export predictions")
    with (ARTIFACT_DIR / name).open("w") as file:
        json.dump({key: [float(value) for value in values] for key, values in history.history.items()}, file, indent=2)


phase1_history = model.fit(
    make_dataset(train_indices, training=True),
    validation_data=make_dataset(validation_indices),
    epochs=PHASE1_EPOCHS, callbacks=training_callbacks(PHASE1_PATH),
    verbose=2, shuffle=False,
)
save_history(phase1_history, "phase1_history.json")
print("Best phase 1 validation macro log2 MSE:", min(phase1_history.history["val_loss"]))
print("Saved phase 1 model:", PHASE1_PATH)


## Phase 2: Fine-tune on the Provided Data

Fine-tuning starts from the best phase 1 weights. It uses only provided-data training rows, and a new Adam optimiser with a lower learning rate so the features learned so far are adjusted rather than overwritten. Early stopping and checkpoint selection use the provided-data validation rows, which match the test distribution. The per-setting weights are recomputed for this subset, so the 9 settings again count equally.


In [ ]:
compile_network(model, FINETUNE_LEARNING_RATE)
finetune_history = model.fit(
    make_dataset(provided_train_indices, training=True),
    validation_data=make_dataset(provided_validation_indices),
    epochs=FINETUNE_EPOCHS, callbacks=training_callbacks(MODEL_PATH),
    verbose=2, shuffle=False,
)
save_history(finetune_history, "finetune_history.json")
print("Best fine-tuning validation macro log2 MSE (provided rows):", min(finetune_history.history["val_loss"]))
print("Saved fine-tuned model:", MODEL_PATH)


## Test the DNN

Inference uses saved models with `training=False`. The only numerical post-processing is `np.exp2`, which converts log₂ predictions to m-heights.

The phase 1 model and the fine-tuned model are both evaluated on the provided-data validation and test rows. This shows what fine-tuning adds. These reports run on the GPU when one is available. The σ cell below repeats the test-row evaluation on the CPU, the path used for submitted predictions.


In [ ]:
def load_prediction_model(model_path=MODEL_PATH, device="/CPU:0"):
    """Load a saved model; return an XLA-compiled prediction function on `device`.

    XLA makes CPU inference about 3x faster for the transformer. Results are deterministic
    and differ from uncompiled inference only in the last float32 digits.
    """
    with tf.device(device):
        network = tf.keras.models.load_model(str(model_path), compile=False)

    @tf.function(jit_compile=True)
    def compiled(batch):
        return network(batch, training=False)

    def predict(batch):
        with tf.device(device):
            return np.asarray(compiled(batch)).reshape(-1).astype(np.float64)

    return predict


def model_log2_batch(predictor, batch):
    values = predictor(batch)
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > MAX_LOG2_HEIGHT):
        raise ValueError("Model returned an invalid log2 m-height; predictions were not saved")
    return values


def predict_heights(network, samples):
    predictions = np.empty(len(samples), dtype=np.float64)
    for start in range(0, len(samples), INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, len(samples))
        log_values = model_log2_batch(network, pack_features(samples[start:stop]))
        predictions[start:stop] = np.exp2(log_values)
    if not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Predictions must be finite and at least 1")
    return predictions


def save_prediction_list(path, predictions):
    predictions = np.asarray(predictions, dtype=np.float64)
    if predictions.ndim != 1 or not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Expected a one-dimensional list of finite predictions >= 1")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as file:
        pickle.dump(predictions.tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(path)


In [ ]:
def report_costs(name, indices, inference_model):
    predicted_log2 = np.empty(len(indices), dtype=np.float64)
    for start in range(0, len(indices), REPORT_BATCH_SIZE):
        selected = indices[start:start + REPORT_BATCH_SIZE]
        batch = {key: values[selected] for key, values in inputs.items()}
        predicted_log2[start:start + len(selected)] = model_log2_batch(inference_model, batch)
    # Use the original float64 labels for reporting the specified cost.
    errors = (np.log2(raw_targets[indices]) - predicted_log2) ** 2
    rows = []
    print(f"\n{name}: per-setting squared log2 error")
    for group_id, setting in enumerate(PARAMETER_SETTINGS):
        mask = setting_ids[indices] == group_id
        cost = float(errors[mask].mean())
        rows.append({"n": setting[0], "k": setting[1], "m": setting[2],
                     "count": int(mask.sum()), "sigma": cost})
        print(f"{setting}: count={mask.sum():,}, sigma={cost:.6f}")
    macro = float(np.mean([row["sigma"] for row in rows]))
    print(f"Equal-setting mean cost: {macro:.6f}")
    print(f"Row-weighted mean cost: {errors.mean():.6f}")
    return {"per_setting": rows, "macro_sigma": macro, "row_mean_sigma": float(errors.mean())}

# Reports run on the GPU when available (much faster for the transformer). The sigma cell
# below re-checks the test rows on the CPU, the path used for submitted predictions.
REPORT_DEVICE = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"
REPORT_BATCH_SIZE = 8192
phase1_model = load_prediction_model(PHASE1_PATH, REPORT_DEVICE)
inference_model = load_prediction_model(MODEL_PATH, REPORT_DEVICE)
cost_report = {
    "phase1": {
        "validation_provided": report_costs("Phase 1, validation (provided rows)", provided_validation_indices, phase1_model),
        "local_test_provided": report_costs("Phase 1, held-out test (provided rows)", provided_test_indices, phase1_model),
    },
    "finetuned": {
        "validation_provided": report_costs("Fine-tuned, validation (provided rows)", provided_validation_indices, inference_model),
        "local_test_provided": report_costs("Fine-tuned, held-out test (provided rows)", provided_test_indices, inference_model),
    },
}
with (ARTIFACT_DIR / "costs.json").open("w") as file:
    json.dump(cost_report, file, indent=2)


## Compute σ from Prediction Files

This cell computes the project's cost exactly as the specification defines it, from three lists:
- the input samples `[n, k, m, P]`;
- the true m-heights;
- the predicted m-heights.

For each parameter setting $(n,k,m)$:

$$\sigma_{n,k,m} = \frac{1}{N_{n,k,m}} \sum_t \left(\log_2 y^{[t]} - \log_2 \hat y^{[t]}\right)^2$$

Here the fine-tuned model's predictions for the provided-data held-out test rows go through `predict_heights` and `save_prediction_list`, are read back from the pickle, and are then scored.


In [ ]:
def compute_sigma(samples, true_heights, predicted_heights):
    true_heights = np.asarray(true_heights, dtype=np.float64)
    predicted_heights = np.asarray(predicted_heights, dtype=np.float64)
    if not len(samples) == len(true_heights) == len(predicted_heights):
        raise ValueError("Samples, true heights and predictions must have the same length")
    if not np.isfinite(predicted_heights).all() or np.any(predicted_heights < 1):
        raise ValueError("Every predicted m-height must be finite and at least 1")
    costs = (np.log2(true_heights) - np.log2(predicted_heights)) ** 2
    settings = [(int(n), int(k), int(m)) for n, k, m, _ in samples]
    sigma = {}
    for setting in PARAMETER_SETTINGS:
        mask = np.array([s == setting for s in settings])
        if mask.any():
            sigma[setting] = float(costs[mask].mean())
            print(f"sigma{setting} = {sigma[setting]:.6f}  (N = {mask.sum():,})")
    print(f"Equal-setting mean sigma: {np.mean(list(sigma.values())):.6f}")
    print(f"Row-weighted mean sigma:  {costs.mean():.6f}")
    return sigma


test_samples = [features[i] for i in provided_test_indices]
test_true_heights = [labels[i] for i in provided_test_indices]
prediction_path = ARTIFACT_DIR / "local_test_predictions.pkl"
cpu_model = load_prediction_model(MODEL_PATH)  # CPU, as in the final cell
save_prediction_list(prediction_path, predict_heights(cpu_model, test_samples))
with prediction_path.open("rb") as file:
    test_predictions = pickle.load(file)
test_sigma = compute_sigma(test_samples, test_true_heights, test_predictions)


## Final Cell: Predict m-Heights for a Test File

This is the cell for testing the submitted model. It is **self-contained**: in a fresh session, run the first cell (it installs the pinned TensorFlow/Keras versions on Colab), then run this cell. None of the training cells or the training data are needed.

1. Place the submitted model file `best_model.keras` next to this notebook. When run from the project folder, the cell also finds it at `artifacts/dnn_v3_bert-small_pretrained/best_model.keras`.
2. Set `TEST_INPUT_PATH` to the test file: a pickled list of samples `[n, k, m, P]`, in the same format as `CSCE-636-Project-1-Train-n_k_m_P`.
3. Run the cell. It writes `OUTPUT_PATH`: a pickled list of predicted m-heights (Python floats), one per input sample in the same order, in the same format as `CSCE-636-Project-1-Train-mHeights`.

The processing is the same as in the evaluation above:
- $G = [I_k \mid P]$ is placed in a zero-padded `6 × 9` array, and `[n, k, m]` is passed alongside it.
- The network outputs $z = \log_2 \hat y$ in `[0, 100]`, and the only post-processing is $\hat y = 2^z$.

Every prediction is therefore finite and at least 1. The cell checks this before saving. Inference runs on the CPU in fixed batches of 1024, compiled with XLA, so repeated runs on a machine give identical results. The transformer predicts about 1,000 rows per second on a multi-core CPU, so a test file with millions of rows takes a while, especially on Colab's two CPU cores.


In [ ]:
# ===== Final cell: load the trained DNN and predict m-heights for a test file =====
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

TEST_INPUT_PATH = "CSCE-636-Project-1-Test-n_k_m_P"   # pickled list of [n, k, m, P]
OUTPUT_PATH = "CSCE-636-Project-1-Test-mHeights"      # pickled list of predicted m-heights
MODEL_CANDIDATES = ["best_model.keras", "artifacts/dnn_v3_bert-small_pretrained/best_model.keras"]
BATCH = 1024


class CompatUnpickler(pickle.Unpickler):
    # Pickles written with NumPy 2 reference numpy._core; NumPy 1.x only has numpy.core.
    def find_class(self, module, name):
        if int(np.__version__.split(".")[0]) < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


if not Path(TEST_INPUT_PATH).exists():
    # Nothing to predict yet (e.g. a batch run before the test file is available).
    print(f"Test file {TEST_INPUT_PATH!r} not found: set TEST_INPUT_PATH and run this cell again.")
else:
    model_file = next((path for path in MODEL_CANDIDATES if Path(path).exists()), None)
    if model_file is None:
        raise FileNotFoundError(f"Trained model not found; looked for {MODEL_CANDIDATES}")
    with tf.device("/CPU:0"):
        network = tf.keras.models.load_model(model_file, compile=False)


    @tf.function(jit_compile=True)
    def predict_log2(batch):
        # XLA-compiled forward pass: about 3x faster on CPU, deterministic for a given machine.
        return network(batch, training=False)


    with open(TEST_INPUT_PATH, "rb") as file:
        samples = CompatUnpickler(file).load()

    # Inputs: G = [I_k | P] in the upper-left corner of a zero-padded 6 x 9 array, and [n, k, m].
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.zeros((len(samples), 3), dtype=np.float32)
    for row, (n, k, m, P) in enumerate(samples):
        P = np.asarray(P, dtype=np.float64)
        if P.shape != (k, n - k):
            raise ValueError(f"Sample {row}: P has shape {P.shape}, expected {(k, n - k)}")
        generators[row, :k, :k] = np.eye(k)
        generators[row, :k, k:n] = P
        parameters[row] = (n, k, m)

    # The network outputs z = log2(m-height); the only post-processing is 2**z.
    log2_heights = np.empty(len(samples), dtype=np.float64)
    with tf.device("/CPU:0"):
        for start in range(0, len(samples), BATCH):
            batch = {"generator": generators[start:start + BATCH],
                     "parameters": parameters[start:start + BATCH]}
            log2_heights[start:start + BATCH] = np.asarray(predict_log2(batch)).reshape(-1)
    predicted_heights = np.exp2(log2_heights)

    if len(predicted_heights) != len(samples) or not np.isfinite(predicted_heights).all() \
            or np.any(predicted_heights < 1):
        raise ValueError("Every prediction must be finite and at least 1; nothing was saved")
    with open(OUTPUT_PATH, "wb") as file:
        pickle.dump(predicted_heights.tolist(), file)

    print(f"Model: {model_file} (TensorFlow {tf.__version__}, Keras {tf.keras.__version__})")
    print(f"Predicted {len(predicted_heights):,} m-heights -> {OUTPUT_PATH}")
    print(f"Range: {predicted_heights.min():.4f} to {predicted_heights.max():.4f}; first five: {predicted_heights[:5].tolist()}")
